In [5]:
import time
import pickle

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from tensorflow.keras.models import Sequential

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

from tensorflow.keras.layers import (
    Embedding,
    Dense,
    SimpleRNN,
    LSTM,
    GRU,
    Bidirectional,
    GlobalAveragePooling1D,
    Dropout,
    BatchNormalization
)

from tensorflow.keras.preprocessing.sequence import pad_sequences

In [9]:
data = pd.read_csv("imdb_cleaned.csv")
print(data.shape)

(49582, 4)


In [10]:
X = data["clean_review"]
y = data["label"]

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_test,
    y_test,
    test_size=0.50,
    random_state=42,
    stratify=y_test
)

print(len(X_train))
print(len(X_val))
print(len(X_test))

39665
4958
4959


In [12]:
import pickle

with open("tokenizer.pkl", "rb") as file:
    tokenizer = pickle.load(file)

print(len(tokenizer.word_index))

90662


In [13]:
MAX_SEQUENCE_LENGTH = 500

X_train_sequences = tokenizer.texts_to_sequences(X_train)
X_val_sequences = tokenizer.texts_to_sequences(X_val)
X_test_sequences = tokenizer.texts_to_sequences(X_test)

X_train_integer = pad_sequences(
    X_train_sequences,
    maxlen=MAX_SEQUENCE_LENGTH,
    padding="post",
    truncating="post"
)

X_val_integer = pad_sequences(
    X_val_sequences,
    maxlen=MAX_SEQUENCE_LENGTH,
    padding="post",
    truncating="post"
)

X_test_integer = pad_sequences(
    X_test_sequences,
    maxlen=MAX_SEQUENCE_LENGTH,
    padding="post",
    truncating="post"
)

y_train = np.asarray(y_train)
y_val = np.asarray(y_val)
y_test = np.asarray(y_test)

print(X_train_integer.shape)
print(X_val_integer.shape)
print(X_test_integer.shape)

(39665, 500)
(4958, 500)
(4959, 500)


In [14]:
def evaluate_model(model, X_test, y_test, model_name, batch_size=64):

    # Generate probabilities
    probabilities = model.predict(
        X_test,
        batch_size=batch_size,
        verbose=0
    ).ravel()

    # Convert probabilities to class predictions
    predictions = (probabilities >= 0.5).astype(int)

    # metrics
    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions, zero_division=0)
    recall = recall_score(y_test,  predictions, zero_division=0)
    f1 = f1_score(y_test, predictions, zero_division=0)
    roc_auc = roc_auc_score(y_test, probabilities)

    # Confusion Matrix
    cm = confusion_matrix(y_test, predictions)

    # Classification Report
    report = classification_report(
        y_test,
        predictions,
        target_names=["Negative", "Positive"],
        digits=4
    )

    # Print results
    print("=" * 60)
    print(f"{model_name} RESULTS")
    print("=" * 60)

    print(f"Accuracy : {accuracy:.5f}")
    print(f"Precision: {precision:.5f}")
    print(f"Recall   : {recall:.5f}")
    print(f"F1 Score : {f1:.5f}")
    print(f"ROC-AUC  : {roc_auc:.5f}")

    print("\nClassification Report")
    print("-" * 60)
    print(report)

    print("Confusion Matrix")
    print(cm)

    # Return everything for later comparison
    results = {
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC-AUC": roc_auc
    }

    return results, probabilities, predictions, cm

In [25]:
lstm_results = []
lstm_baseline_results = {
    "Model": "LSTM Baseline",
    "Accuracy": 0.88264,
    "Precision": 0.87913,
    "Recall": 0.88831,
    "F1 Score": 0.88369,
    "ROC-AUC": 0.94016
}

lstm_results.append(lstm_baseline_results)
lstm_results

[{'Model': 'LSTM Baseline',
  'Accuracy': 0.88264,
  'Precision': 0.87913,
  'Recall': 0.88831,
  'F1 Score': 0.88369,
  'ROC-AUC': 0.94016}]

**LSTM(EarlyStopping)**

In [17]:
from tensorflow.keras import Sequential, Input

In [15]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

In [53]:
VOCAB_SIZE = 30000
EMBEDDING_DIM = 128
MAX_SEQUENCE_LENGTH = 500

In [20]:

lstm_early_model = Sequential([
    Input(shape=(MAX_SEQUENCE_LENGTH,), dtype="int32"),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    LSTM(128),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])
lstm_early_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
lstm_early_model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 500, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,979,905 (15.18 MB)

 Trainable params: 3,979,905 (15.18 MB)

 Non-trainable params: 0 (0.00 B)

In [23]:
history_lstm_early = lstm_early_model.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=15,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/15
620/620 ━━━━━━━━━━━━━━━━━━━━ 24s 29ms/step - accuracy: 0.4997 - loss: 0.6930 - val_accuracy: 0.5056 - val_loss: 0.6914
Epoch 2/15
620/620 ━━━━━━━━━━━━━━━━━━━━ 18s 28ms/step - accuracy: 0.5203 - loss: 0.6824 - val_accuracy: 0.5123 - val_loss: 0.6888
Epoch 3/15
620/620 ━━━━━━━━━━━━━━━━━━━━ 31s 50ms/step - accuracy: 0.5328 - loss: 0.6561 - val_accuracy: 0.5111 - val_loss: 0.7100
Epoch 4/15
620/620 ━━━━━━━━━━━━━━━━━━━━ 28s 30ms/step - accuracy: 0.5411 - loss: 0.6425 - val_accuracy: 0.5131 - val_loss: 0.7501
Epoch 5/15
620/620 ━━━━━━━━━━━━━━━━━━━━ 21s 34ms/step - accuracy: 0.7436 - loss: 0.4770 - val_accuracy: 0.8558 - val_loss: 0.3674
Epoch 6/15
620/620 ━━━━━━━━━━━━━━━━━━━━ 22s 35ms/step - accuracy: 0.9037 - loss: 0.2484 - val_accuracy: 0.8810 - val_loss: 0.3113
Epoch 7/15
620/620 ━━━━━━━━━━━━━━━━━━━━ 20s 32ms/step - accuracy: 0.9557 - loss: 0.1335 - val_accuracy: 0.8743 - val_loss: 0.3582
Epoch 8/15
620/620 ━━━━━━━━━━━━━━━━━━━━ 23s 37ms/step - accuracy: 0.9759 - loss: 0.0782 - 

In [24]:
lstm_early_results, lstm_early_probabilities, lstm_early_predictions, lstm_early_cm = evaluate_model(
    model=lstm_early_model,
    X_test=X_test_integer,
    y_test=y_test,
    model_name="LSTM EarlyStopping",
    batch_size=64
)

LSTM EarlyStopping RESULTS
Accuracy : 0.88022
Precision: 0.88689
Recall   : 0.87264
F1 Score : 0.87971
ROC-AUC  : 0.94813

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.8737    0.8879    0.8807      2470
    Positive     0.8869    0.8726    0.8797      2489

    accuracy                         0.8802      4959
   macro avg     0.8803    0.8802    0.8802      4959
weighted avg     0.8803    0.8802    0.8802      4959

Confusion Matrix
[[2193  277]
 [ 317 2172]]


In [26]:
lstm_results.append(lstm_early_results)

lstm_df = pd.DataFrame(lstm_results)
lstm_df

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,LSTM Baseline,0.882640,0.879130,0.88831,0.883690,0.940160
1,LSTM EarlyStopping,0.880218,0.886893,0.87264,0.879708,0.948134


**LSTM(DROPOUT)**

In [27]:
lstm_dropout_model = Sequential([
    Input(shape=(MAX_SEQUENCE_LENGTH,), dtype="int32"),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    LSTM(128),
    Dropout(0.5),
    Dense(64, activation="relu"),
    Dropout(0.5),
    Dense(1, activation="sigmoid")
])

lstm_dropout_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

lstm_dropout_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 500, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,979,905 (15.18 MB)

 Trainable params: 3,979,905 (15.18 MB)

 Non-trainable params: 0 (0.00 B)

In [28]:
history_lstm_dropout = lstm_dropout_model.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 30s 43ms/step - accuracy: 0.4995 - loss: 0.6938 - val_accuracy: 0.5018 - val_loss: 0.6928
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 22s 36ms/step - accuracy: 0.5006 - loss: 0.6927 - val_accuracy: 0.5014 - val_loss: 0.6948
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 37s 30ms/step - accuracy: 0.5184 - loss: 0.6801 - val_accuracy: 0.5139 - val_loss: 0.6915
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 22s 35ms/step - accuracy: 0.5325 - loss: 0.6554 - val_accuracy: 0.5141 - val_loss: 0.7203
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 24s 39ms/step - accuracy: 0.5419 - loss: 0.6428 - val_accuracy: 0.5133 - val_loss: 0.7608
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 21s 34ms/step - accuracy: 0.5413 - loss: 0.6390 - val_accuracy: 0.5075 - val_loss: 0.7982
Epoch 7/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 27s 44ms/step - accuracy: 0.5436 - loss: 0.6385 - val_accuracy: 0.5079 - val_loss: 0.8053
Epoch 8/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 24s 38ms/step - accuracy: 0.5415 - loss: 0.6374 - 

In [29]:
lstm_dropout_results, lstm_dropout_probabilities, lstm_dropout_predictions, lstm_dropout_cm = evaluate_model(
    lstm_dropout_model,
    X_test_integer,
    y_test,
    "LSTM Dropout",
    batch_size=64
)

LSTM Dropout RESULTS
Accuracy : 0.75156
Precision: 0.88629
Recall   : 0.57935
F1 Score : 0.70068
ROC-AUC  : 0.83426

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.6858    0.9251    0.7877      2470
    Positive     0.8863    0.5793    0.7007      2489

    accuracy                         0.7516      4959
   macro avg     0.7860    0.7522    0.7442      4959
weighted avg     0.7864    0.7516    0.7440      4959

Confusion Matrix
[[2285  185]
 [1047 1442]]


In [30]:
lstm_results.append(lstm_dropout_results)
lstm_df = pd.DataFrame(lstm_results)
display(lstm_df)


,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,LSTM Baseline,0.882640,0.879130,0.888310,0.883690,0.940160
1,LSTM EarlyStopping,0.880218,0.886893,0.872640,0.879708,0.948134
2,LSTM Dropout,0.751563,0.886294,0.579349,0.700680,0.834259


**LSTM(BATCHNORMALIZATION)**

In [31]:
lstm_batchnorm_model = Sequential([
    Input(
        shape=(MAX_SEQUENCE_LENGTH,),
        dtype="int32"
    ),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    LSTM(128),
    BatchNormalization(),
    Dense(64, activation="relu"),
    BatchNormalization(),
    Dense(1, activation="sigmoid")
])

lstm_batchnorm_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

lstm_batchnorm_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 500, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,980,673 (15.19 MB)

 Trainable params: 3,980,289 (15.18 MB)

 Non-trainable params: 384 (1.50 KB)

In [32]:
history_lstm_batchnorm = lstm_batchnorm_model.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 27s 38ms/step - accuracy: 0.5020 - loss: 0.7000 - val_accuracy: 0.5018 - val_loss: 3.1709
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 32s 51ms/step - accuracy: 0.5552 - loss: 0.6638 - val_accuracy: 0.5018 - val_loss: 5.5709
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 31s 35ms/step - accuracy: 0.5901 - loss: 0.6421 - val_accuracy: 0.5018 - val_loss: 1.1215
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 24s 39ms/step - accuracy: 0.7573 - loss: 0.4591 - val_accuracy: 0.8132 - val_loss: 0.4089
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 47s 48ms/step - accuracy: 0.9114 - loss: 0.2141 - val_accuracy: 0.5468 - val_loss: 2.0774
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 23s 37ms/step - accuracy: 0.9488 - loss: 0.1304 - val_accuracy: 0.8282 - val_loss: 0.4740
Epoch 7/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 25s 41ms/step - accuracy: 0.9644 - loss: 0.0890 - val_accuracy: 0.8173 - val_loss: 0.5745
Epoch 8/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 28s 45ms/step - accuracy: 0.9806 - loss: 0.0552 - 

In [33]:
lstm_batchnorm_results, lstm_batchnorm_probabilities, lstm_batchnorm_predictions, lstm_batchnorm_cm = evaluate_model(
    lstm_batchnorm_model,
    X_test_integer,
    y_test,
    "LSTM Batch Normalization",
    batch_size=64
)

LSTM Batch Normalization RESULTS
Accuracy : 0.81791
Precision: 0.74415
Recall   : 0.97107
F1 Score : 0.84260
ROC-AUC  : 0.94595

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.9579    0.6636    0.7840      2470
    Positive     0.7442    0.9711    0.8426      2489

    accuracy                         0.8179      4959
   macro avg     0.8510    0.8173    0.8133      4959
weighted avg     0.8506    0.8179    0.8134      4959

Confusion Matrix
[[1639  831]
 [  72 2417]]


In [34]:
lstm_results.append(lstm_batchnorm_results)

lstm_df = pd.DataFrame(lstm_results)
display(lstm_df)


,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,LSTM Baseline,0.882640,0.879130,0.888310,0.883690,0.940160
1,LSTM EarlyStopping,0.880218,0.886893,0.872640,0.879708,0.948134
2,LSTM Dropout,0.751563,0.886294,0.579349,0.700680,0.834259
3,LSTM Batch Normalization,0.817907,0.744150,0.971073,0.842601,0.945947


**LSTM(Learning Rate)**

In [35]:
lstm_lr_model = Sequential([
    Input(
        shape=(MAX_SEQUENCE_LENGTH,),
        dtype="int32"
    ),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    LSTM(128),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

lstm_lr_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

lstm_lr_model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 500, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,979,905 (15.18 MB)

 Trainable params: 3,979,905 (15.18 MB)

 Non-trainable params: 0 (0.00 B)

In [36]:
history_lstm_lr = lstm_lr_model.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 22s 33ms/step - accuracy: 0.5041 - loss: 0.6931 - val_accuracy: 0.5018 - val_loss: 0.6926
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 19s 31ms/step - accuracy: 0.5166 - loss: 0.6844 - val_accuracy: 0.5190 - val_loss: 0.6848
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 26s 42ms/step - accuracy: 0.5282 - loss: 0.6604 - val_accuracy: 0.5131 - val_loss: 0.6981
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 19s 31ms/step - accuracy: 0.5373 - loss: 0.6435 - val_accuracy: 0.5186 - val_loss: 0.7315
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 21s 34ms/step - accuracy: 0.5379 - loss: 0.6401 - val_accuracy: 0.5165 - val_loss: 0.7558
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 29s 47ms/step - accuracy: 0.5413 - loss: 0.6612 - val_accuracy: 0.5167 - val_loss: 0.7451
Epoch 7/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 39s 44ms/step - accuracy: 0.5421 - loss: 0.6396 - val_accuracy: 0.5169 - val_loss: 0.7421
Epoch 8/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 20s 32ms/step - accuracy: 0.5428 - loss: 0.6381 - 

In [37]:
lstm_lr_results, lstm_lr_probabilities, lstm_lr_predictions, lstm_lr_cm = evaluate_model(
    lstm_lr_model,
    X_test_integer,
    y_test,
    "LSTM Learning Rate 0.0005",
    batch_size=64
)

LSTM Learning Rate 0.0005 RESULTS
Accuracy : 0.85602
Precision: 0.86598
Recall   : 0.84371
F1 Score : 0.85470
ROC-AUC  : 0.91698

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.8465    0.8684    0.8573      2470
    Positive     0.8660    0.8437    0.8547      2489

    accuracy                         0.8560      4959
   macro avg     0.8562    0.8561    0.8560      4959
weighted avg     0.8563    0.8560    0.8560      4959

Confusion Matrix
[[2145  325]
 [ 389 2100]]


In [38]:
lstm_results.append(lstm_lr_results)
lstm_df = pd.DataFrame(lstm_results)
display(lstm_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,LSTM Baseline,0.882640,0.879130,0.888310,0.883690,0.940160
1,LSTM EarlyStopping,0.880218,0.886893,0.872640,0.879708,0.948134
2,LSTM Dropout,0.751563,0.886294,0.579349,0.700680,0.834259
3,LSTM Batch Normalization,0.817907,0.744150,0.971073,0.842601,0.945947
4,LSTM Learning Rate 0.0005,0.856019,0.865979,0.843712,0.854701,0.916978


**LSTM ReduceLR**

In [40]:
lstm_reducelr_model = Sequential([
    Input(
        shape=(MAX_SEQUENCE_LENGTH,),
        dtype="int32"
    ),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    LSTM(128),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

lstm_reducelr_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

lstm_reducelr_model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (None, 500, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,979,905 (15.18 MB)

 Trainable params: 3,979,905 (15.18 MB)

 Non-trainable params: 0 (0.00 B)

In [41]:
history_lstm_reducelr = lstm_reducelr_model.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=64,
    callbacks=[reduce_lr],
    verbose=1
)

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 22s 34ms/step - accuracy: 0.4998 - loss: 0.6940 - val_accuracy: 0.5063 - val_loss: 0.6907 - learning_rate: 0.0010
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 18s 29ms/step - accuracy: 0.5183 - loss: 0.6809 - val_accuracy: 0.5153 - val_loss: 0.6888 - learning_rate: 0.0010
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 21s 30ms/step - accuracy: 0.5302 - loss: 0.6545 - val_accuracy: 0.5103 - val_loss: 0.7056 - learning_rate: 0.0010
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.5331 - loss: 0.6418
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
620/620 ━━━━━━━━━━━━━━━━━━━━ 18s 29ms/step - accuracy: 0.5384 - loss: 0.6426 - val_accuracy: 0.5157 - val_loss: 0.7474 - learning_rate: 0.0010
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 29s 46ms/step - accuracy: 0.6004 - loss: 0.6106 - val_accuracy: 0.7428 - val_loss: 0.5823 - learning_rate: 5.0000e-04
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 31s 30ms/step - accuracy: 0.8659 -

In [42]:
lstm_reducelr_results, lstm_reducelr_probabilities, lstm_reducelr_predictions, lstm_reducelr_cm = evaluate_model(
    lstm_reducelr_model,
    X_test_integer,
    y_test,
    "LSTM ReduceLR",
    batch_size=64
)

LSTM ReduceLR RESULTS
Accuracy : 0.88425
Precision: 0.88531
Recall   : 0.88389
F1 Score : 0.88460
ROC-AUC  : 0.94395

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.8832    0.8846    0.8839      2470
    Positive     0.8853    0.8839    0.8846      2489

    accuracy                         0.8843      4959
   macro avg     0.8842    0.8843    0.8842      4959
weighted avg     0.8843    0.8843    0.8843      4959

Confusion Matrix
[[2185  285]
 [ 289 2200]]


In [43]:
lstm_results.append(lstm_reducelr_results)
lstm_df = pd.DataFrame(lstm_results)
display(lstm_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,LSTM Baseline,0.882640,0.879130,0.888310,0.883690,0.940160
1,LSTM EarlyStopping,0.880218,0.886893,0.872640,0.879708,0.948134
2,LSTM Dropout,0.751563,0.886294,0.579349,0.700680,0.834259
3,LSTM Batch Normalization,0.817907,0.744150,0.971073,0.842601,0.945947
4,LSTM Learning Rate 0.0005,0.856019,0.865979,0.843712,0.854701,0.916978
5,LSTM ReduceLR,0.884251,0.885312,0.883889,0.884600,0.943946


**LSTM(Batchsize 32)**

In [44]:
lstm_batch32_model = Sequential([
    Input(
        shape=(MAX_SEQUENCE_LENGTH,),
        dtype="int32"
    ),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    LSTM(128),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

lstm_batch32_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

lstm_batch32_model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ (None, 500, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,979,905 (15.18 MB)

 Trainable params: 3,979,905 (15.18 MB)

 Non-trainable params: 0 (0.00 B)

In [45]:
history_lstm_batch32 = lstm_batch32_model.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=32,
    verbose=1
)

Epoch 1/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 48s 37ms/step - accuracy: 0.5052 - loss: 0.6935 - val_accuracy: 0.5101 - val_loss: 0.6922
Epoch 2/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 32s 26ms/step - accuracy: 0.5205 - loss: 0.6832 - val_accuracy: 0.5139 - val_loss: 0.6879
Epoch 3/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 38s 30ms/step - accuracy: 0.5431 - loss: 0.6528 - val_accuracy: 0.7408 - val_loss: 0.5997
Epoch 4/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 48s 36ms/step - accuracy: 0.8365 - loss: 0.3940 - val_accuracy: 0.8695 - val_loss: 0.3403
Epoch 5/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 44s 36ms/step - accuracy: 0.9244 - loss: 0.2075 - val_accuracy: 0.8786 - val_loss: 0.3197
Epoch 6/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 34s 27ms/step - accuracy: 0.9613 - loss: 0.1185 - val_accuracy: 0.8725 - val_loss: 0.3885
Epoch 7/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 43s 35ms/step - accuracy: 0.9803 - loss: 0.0659 - val_accuracy: 0.8616 - val_loss: 0.4815
Epoch 8/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 46s 37ms/step - accuracy: 0.9889 -

In [46]:
lstm_batch32_results, lstm_batch32_probabilities, lstm_batch32_predictions, lstm_batch32_cm = evaluate_model(lstm_batch32_model,
    X_test_integer,
    y_test,
    "LSTM Batch Size 32",
    batch_size=32
)

LSTM Batch Size 32 RESULTS
Accuracy : 0.87477
Precision: 0.85139
Recall   : 0.90920
F1 Score : 0.87935
ROC-AUC  : 0.93263

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.9018    0.8401    0.8698      2470
    Positive     0.8514    0.9092    0.8793      2489

    accuracy                         0.8748      4959
   macro avg     0.8766    0.8746    0.8746      4959
weighted avg     0.8765    0.8748    0.8746      4959

Confusion Matrix
[[2075  395]
 [ 226 2263]]


In [47]:
lstm_results.append(lstm_batch32_results)
lstm_df = pd.DataFrame(lstm_results)

display(lstm_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,LSTM Baseline,0.882640,0.879130,0.888310,0.883690,0.940160
1,LSTM EarlyStopping,0.880218,0.886893,0.872640,0.879708,0.948134
2,LSTM Dropout,0.751563,0.886294,0.579349,0.700680,0.834259
3,LSTM Batch Normalization,0.817907,0.744150,0.971073,0.842601,0.945947
4,LSTM Learning Rate 0.0005,0.856019,0.865979,0.843712,0.854701,0.916978
5,LSTM ReduceLR,0.884251,0.885312,0.883889,0.884600,0.943946
6,LSTM Batch Size 32,0.874773,0.851392,0.909200,0.879347,0.932626


**LSTM(Batch Size 128)**

In [48]:
lstm_batch128_model = Sequential([
    Input(
        shape=(MAX_SEQUENCE_LENGTH,),
        dtype="int32"
    ),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    LSTM(128),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

lstm_batch128_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

lstm_batch128_model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_6 (Embedding)         │ (None, 500, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_6 (LSTM)                   │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,979,905 (15.18 MB)

 Trainable params: 3,979,905 (15.18 MB)

 Non-trainable params: 0 (0.00 B)

In [49]:
history_lstm_batch128 = lstm_batch128_model.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=128,
    verbose=1
)

Epoch 1/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 15s 43ms/step - accuracy: 0.5020 - loss: 0.6936 - val_accuracy: 0.5093 - val_loss: 0.6910
Epoch 2/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 13s 42ms/step - accuracy: 0.5188 - loss: 0.6807 - val_accuracy: 0.5046 - val_loss: 0.6933
Epoch 3/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 13s 43ms/step - accuracy: 0.5315 - loss: 0.6564 - val_accuracy: 0.5087 - val_loss: 0.7047
Epoch 4/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 13s 43ms/step - accuracy: 0.5314 - loss: 0.6444 - val_accuracy: 0.5119 - val_loss: 0.7373
Epoch 5/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 21s 43ms/step - accuracy: 0.5827 - loss: 0.6269 - val_accuracy: 0.5186 - val_loss: 0.7149
Epoch 6/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 13s 43ms/step - accuracy: 0.7853 - loss: 0.4307 - val_accuracy: 0.8622 - val_loss: 0.3445
Epoch 7/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 14s 45ms/step - accuracy: 0.9235 - loss: 0.2049 - val_accuracy: 0.8739 - val_loss: 0.3398
Epoch 8/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 13s 43ms/step - accuracy: 0.9612 - loss: 0.1223 - 

In [50]:
lstm_batch128_results, lstm_batch128_probabilities, lstm_batch128_predictions, lstm_batch128_cm = evaluate_model(
    lstm_batch128_model,
    X_test_integer,
    y_test,
    "LSTM Batch Size 128",
    batch_size=128
)

LSTM Batch Size 128 RESULTS
Accuracy : 0.88183
Precision: 0.88136
Recall   : 0.88349
F1 Score : 0.88242
ROC-AUC  : 0.94074

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.8823    0.8802    0.8812      2470
    Positive     0.8814    0.8835    0.8824      2489

    accuracy                         0.8818      4959
   macro avg     0.8818    0.8818    0.8818      4959
weighted avg     0.8818    0.8818    0.8818      4959

Confusion Matrix
[[2174  296]
 [ 290 2199]]


In [51]:
lstm_results.append(lstm_batch128_results)
lstm_df = pd.DataFrame(lstm_results)
display(lstm_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,LSTM Baseline,0.882640,0.879130,0.888310,0.883690,0.940160
1,LSTM EarlyStopping,0.880218,0.886893,0.872640,0.879708,0.948134
2,LSTM Dropout,0.751563,0.886294,0.579349,0.700680,0.834259
3,LSTM Batch Normalization,0.817907,0.744150,0.971073,0.842601,0.945947
4,LSTM Learning Rate 0.0005,0.856019,0.865979,0.843712,0.854701,0.916978
5,LSTM ReduceLR,0.884251,0.885312,0.883889,0.884600,0.943946
6,LSTM Batch Size 32,0.874773,0.851392,0.909200,0.879347,0.932626
7,LSTM Batch Size 128,0.881831,0.881363,0.883487,0.882424,0.940744


**LSTM(SGD)**

In [54]:
lstm_sgd_model = Sequential([
    Input(
        shape=(MAX_SEQUENCE_LENGTH,),
        dtype="int32"
    ),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    LSTM(128),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

lstm_sgd_model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

lstm_sgd_model.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_7 (Embedding)         │ (None, 500, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_7 (LSTM)                   │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,979,905 (15.18 MB)

 Trainable params: 3,979,905 (15.18 MB)

 Non-trainable params: 0 (0.00 B)

In [55]:
history_lstm_sgd = lstm_sgd_model.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 17s 26ms/step - accuracy: 0.4970 - loss: 0.6932 - val_accuracy: 0.4978 - val_loss: 0.6931
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 16s 26ms/step - accuracy: 0.4980 - loss: 0.6932 - val_accuracy: 0.4994 - val_loss: 0.6931
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 16s 26ms/step - accuracy: 0.4979 - loss: 0.6932 - val_accuracy: 0.5006 - val_loss: 0.6931
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 16s 26ms/step - accuracy: 0.4993 - loss: 0.6932 - val_accuracy: 0.5026 - val_loss: 0.6931
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 16s 25ms/step - accuracy: 0.4996 - loss: 0.6931 - val_accuracy: 0.5030 - val_loss: 0.6931
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 21s 26ms/step - accuracy: 0.5012 - loss: 0.6931 - val_accuracy: 0.5030 - val_loss: 0.6931
Epoch 7/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 16s 26ms/step - accuracy: 0.5018 - loss: 0.6931 - val_accuracy: 0.5034 - val_loss: 0.6931
Epoch 8/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 16s 26ms/step - accuracy: 0.5018 - loss: 0.6931 - 

In [56]:
lstm_sgd_results, lstm_sgd_probabilities, lstm_sgd_predictions, lstm_sgd_cm = evaluate_model(
    lstm_sgd_model,
    X_test_integer,
    y_test,
    "LSTM SGD",
    batch_size=64
)

LSTM SGD RESULTS
Accuracy : 0.50192
Precision: 0.50192
Recall   : 0.99679
F1 Score : 0.66765
ROC-AUC  : 0.51380

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.5000    0.0032    0.0064      2470
    Positive     0.5019    0.9968    0.6677      2489

    accuracy                         0.5019      4959
   macro avg     0.5010    0.5000    0.3370      4959
weighted avg     0.5010    0.5019    0.3383      4959

Confusion Matrix
[[   8 2462]
 [   8 2481]]


In [57]:
lstm_results.append(lstm_sgd_results)

lstm_df = pd.DataFrame(lstm_results)
display(lstm_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,LSTM Baseline,0.882640,0.879130,0.888310,0.883690,0.940160
1,LSTM EarlyStopping,0.880218,0.886893,0.872640,0.879708,0.948134
2,LSTM Dropout,0.751563,0.886294,0.579349,0.700680,0.834259
3,LSTM Batch Normalization,0.817907,0.744150,0.971073,0.842601,0.945947
4,LSTM Learning Rate 0.0005,0.856019,0.865979,0.843712,0.854701,0.916978
5,LSTM ReduceLR,0.884251,0.885312,0.883889,0.884600,0.943946
6,LSTM Batch Size 32,0.874773,0.851392,0.909200,0.879347,0.932626
7,LSTM Batch Size 128,0.881831,0.881363,0.883487,0.882424,0.940744
8,LSTM SGD,0.501916,0.501922,0.996786,0.667653,0.513804


**LSTM(RMSprop)**

In [58]:
lstm_rmsprop_model = Sequential([
    Input(
        shape=(MAX_SEQUENCE_LENGTH,),
        dtype="int32"
    ),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    LSTM(128),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

lstm_rmsprop_model.compile(optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

lstm_rmsprop_model.summary()

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_8 (Embedding)         │ (None, 500, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_8 (LSTM)                   │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,979,905 (15.18 MB)

 Trainable params: 3,979,905 (15.18 MB)

 Non-trainable params: 0 (0.00 B)

In [59]:
history_lstm_rmsprop = lstm_rmsprop_model.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 19s 28ms/step - accuracy: 0.5024 - loss: 0.6933 - val_accuracy: 0.5052 - val_loss: 0.6929
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 18s 29ms/step - accuracy: 0.5027 - loss: 0.6930 - val_accuracy: 0.5058 - val_loss: 0.6926
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 18s 29ms/step - accuracy: 0.5037 - loss: 0.6922 - val_accuracy: 0.5097 - val_loss: 0.6909
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 18s 28ms/step - accuracy: 0.5047 - loss: 0.6923 - val_accuracy: 0.5083 - val_loss: 0.6931
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 18s 29ms/step - accuracy: 0.5030 - loss: 0.6931 - val_accuracy: 0.5018 - val_loss: 0.6931
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 17s 28ms/step - accuracy: 0.5037 - loss: 0.6921 - val_accuracy: 0.5133 - val_loss: 0.6882
Epoch 7/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 18s 29ms/step - accuracy: 0.5041 - loss: 0.6913 - val_accuracy: 0.4982 - val_loss: 0.6932
Epoch 8/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 17s 28ms/step - accuracy: 0.5038 - loss: 0.6916 - 

In [60]:
lstm_rmsprop_results, lstm_rmsprop_probabilities, lstm_rmsprop_predictions, lstm_rmsprop_cm = evaluate_model(
    lstm_rmsprop_model,
    X_test_integer,
    y_test,
    "LSTM RMSprop",
    batch_size=64
)

LSTM RMSprop RESULTS
Accuracy : 0.51704
Precision: 0.71560
Recall   : 0.06268
F1 Score : 0.11526
ROC-AUC  : 0.52904

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.5079    0.9749    0.6679      2470
    Positive     0.7156    0.0627    0.1153      2489

    accuracy                         0.5170      4959
   macro avg     0.6118    0.5188    0.3916      4959
weighted avg     0.6122    0.5170    0.3905      4959

Confusion Matrix
[[2408   62]
 [2333  156]]


In [61]:
lstm_results.append(lstm_rmsprop_results)

lstm_df = pd.DataFrame(lstm_results)
display(lstm_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,LSTM Baseline,0.882640,0.879130,0.888310,0.883690,0.940160
1,LSTM EarlyStopping,0.880218,0.886893,0.872640,0.879708,0.948134
2,LSTM Dropout,0.751563,0.886294,0.579349,0.700680,0.834259
3,LSTM Batch Normalization,0.817907,0.744150,0.971073,0.842601,0.945947
4,LSTM Learning Rate 0.0005,0.856019,0.865979,0.843712,0.854701,0.916978
5,LSTM ReduceLR,0.884251,0.885312,0.883889,0.884600,0.943946
6,LSTM Batch Size 32,0.874773,0.851392,0.909200,0.879347,0.932626
7,LSTM Batch Size 128,0.881831,0.881363,0.883487,0.882424,0.940744
8,LSTM SGD,0.501916,0.501922,0.996786,0.667653,0.513804
9,LSTM RMSprop,0.517040,0.715596,0.062676,0.115257,0.529041


**LSTM(Dim 128)**

In [62]:
LSTM_DIM = 128

lstm_dim128_model = Sequential([
    Input(
        shape=(MAX_SEQUENCE_LENGTH,),
        dtype="int32"
    ),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=LSTM_DIM
    ),
    LSTM(128),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])
lstm_dim128_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

lstm_dim128_model.summary()

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_9 (Embedding)         │ (None, 500, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_9 (LSTM)                   │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,979,905 (15.18 MB)

 Trainable params: 3,979,905 (15.18 MB)

 Non-trainable params: 0 (0.00 B)

In [63]:
history_lstm_dim128 = lstm_dim128_model.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 19s 29ms/step - accuracy: 0.5029 - loss: 0.6945 - val_accuracy: 0.5085 - val_loss: 0.6924
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 18s 29ms/step - accuracy: 0.5191 - loss: 0.6845 - val_accuracy: 0.5133 - val_loss: 0.6906
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 18s 28ms/step - accuracy: 0.5333 - loss: 0.6614 - val_accuracy: 0.5083 - val_loss: 0.7026
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 18s 29ms/step - accuracy: 0.6024 - loss: 0.6245 - val_accuracy: 0.7810 - val_loss: 0.5003
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 18s 29ms/step - accuracy: 0.8628 - loss: 0.3294 - val_accuracy: 0.8788 - val_loss: 0.3030
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 18s 29ms/step - accuracy: 0.9338 - loss: 0.1821 - val_accuracy: 0.8832 - val_loss: 0.3124
Epoch 7/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 18s 29ms/step - accuracy: 0.9603 - loss: 0.1207 - val_accuracy: 0.8800 - val_loss: 0.3394
Epoch 8/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 18s 29ms/step - accuracy: 0.9755 - loss: 0.0804 - 

In [64]:
lstm_dim128_results, lstm_dim128_probabilities, lstm_dim128_predictions, lstm_dim128_cm = evaluate_model(
    lstm_dim128_model,
    X_test_integer,
    y_test,
    "LSTM Dim 128",
    batch_size=64
)

LSTM Dim 128 RESULTS
Accuracy : 0.87760
Precision: 0.88036
Recall   : 0.87505
F1 Score : 0.87769
ROC-AUC  : 0.94122

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.8748    0.8802    0.8775      2470
    Positive     0.8804    0.8751    0.8777      2489

    accuracy                         0.8776      4959
   macro avg     0.8776    0.8776    0.8776      4959
weighted avg     0.8776    0.8776    0.8776      4959

Confusion Matrix
[[2174  296]
 [ 311 2178]]


In [65]:
lstm_results.append(lstm_dim128_results)

lstm_df = pd.DataFrame(lstm_results)
display(lstm_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,LSTM Baseline,0.882640,0.879130,0.888310,0.883690,0.940160
1,LSTM EarlyStopping,0.880218,0.886893,0.872640,0.879708,0.948134
2,LSTM Dropout,0.751563,0.886294,0.579349,0.700680,0.834259
3,LSTM Batch Normalization,0.817907,0.744150,0.971073,0.842601,0.945947
4,LSTM Learning Rate 0.0005,0.856019,0.865979,0.843712,0.854701,0.916978
5,LSTM ReduceLR,0.884251,0.885312,0.883889,0.884600,0.943946
6,LSTM Batch Size 32,0.874773,0.851392,0.909200,0.879347,0.932626
7,LSTM Batch Size 128,0.881831,0.881363,0.883487,0.882424,0.940744
8,LSTM SGD,0.501916,0.501922,0.996786,0.667653,0.513804
9,LSTM RMSprop,0.517040,0.715596,0.062676,0.115257,0.529041


**LSTM Dim 256**

In [66]:
lstm_dim256_model = Sequential([
    Input(
        shape=(MAX_SEQUENCE_LENGTH,),
        dtype="int32"
    ),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    LSTM(256),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])


lstm_dim256_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

lstm_dim256_model.summary()

Model: "sequential_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_10 (Embedding)        │ (None, 500, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_10 (LSTM)                  │ (None, 256)            │       394,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,250,753 (16.22 MB)

 Trainable params: 4,250,753 (16.22 MB)

 Non-trainable params: 0 (0.00 B)

In [67]:
history_lstm_dim256 = lstm_dim256_model.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 32s 49ms/step - accuracy: 0.5057 - loss: 0.6942 - val_accuracy: 0.5077 - val_loss: 0.6921
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 30s 48ms/step - accuracy: 0.5140 - loss: 0.6853 - val_accuracy: 0.5105 - val_loss: 0.6905
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 31s 50ms/step - accuracy: 0.5317 - loss: 0.6639 - val_accuracy: 0.5137 - val_loss: 0.7143
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 30s 49ms/step - accuracy: 0.5381 - loss: 0.6455 - val_accuracy: 0.5117 - val_loss: 0.7487
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 30s 49ms/step - accuracy: 0.6557 - loss: 0.5762 - val_accuracy: 0.8118 - val_loss: 0.4754
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 31s 50ms/step - accuracy: 0.8708 - loss: 0.3102 - val_accuracy: 0.8814 - val_loss: 0.3400
Epoch 7/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 31s 50ms/step - accuracy: 0.9461 - loss: 0.1543 - val_accuracy: 0.8885 - val_loss: 0.3447
Epoch 8/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 31s 50ms/step - accuracy: 0.9739 - loss: 0.0859 - 

In [68]:
lstm_dim256_results, lstm_dim256_probabilities, lstm_dim256_predictions, lstm_dim256_cm = evaluate_model(
    lstm_dim256_model,
    X_test_integer,
    y_test,
    "LSTM Dim 256",
    batch_size=64
)

LSTM Dim 256 RESULTS
Accuracy : 0.88183
Precision: 0.87653
Recall   : 0.88992
F1 Score : 0.88317
ROC-AUC  : 0.93789

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.8873    0.8737    0.8805      2470
    Positive     0.8765    0.8899    0.8832      2489

    accuracy                         0.8818      4959
   macro avg     0.8819    0.8818    0.8818      4959
weighted avg     0.8819    0.8818    0.8818      4959

Confusion Matrix
[[2158  312]
 [ 274 2215]]


In [69]:
lstm_results.append(lstm_dim256_results)

lstm_df = pd.DataFrame(lstm_results)
display(lstm_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,LSTM Baseline,0.882640,0.879130,0.888310,0.883690,0.940160
1,LSTM EarlyStopping,0.880218,0.886893,0.872640,0.879708,0.948134
2,LSTM Dropout,0.751563,0.886294,0.579349,0.700680,0.834259
3,LSTM Batch Normalization,0.817907,0.744150,0.971073,0.842601,0.945947
4,LSTM Learning Rate 0.0005,0.856019,0.865979,0.843712,0.854701,0.916978
5,LSTM ReduceLR,0.884251,0.885312,0.883889,0.884600,0.943946
6,LSTM Batch Size 32,0.874773,0.851392,0.909200,0.879347,0.932626
7,LSTM Batch Size 128,0.881831,0.881363,0.883487,0.882424,0.940744
8,LSTM SGD,0.501916,0.501922,0.996786,0.667653,0.513804
9,LSTM RMSprop,0.517040,0.715596,0.062676,0.115257,0.529041


**LSTM Sequence Length 300**

In [70]:
MAX_SEQUENCE_LENGTH_300 = 300

X_train_seq300 = tokenizer.texts_to_sequences(X_train)
X_val_seq300   = tokenizer.texts_to_sequences(X_val)
X_test_seq300  = tokenizer.texts_to_sequences(X_test)

X_train_integer_300 = pad_sequences(
    X_train_seq300,
    maxlen=MAX_SEQUENCE_LENGTH_300,
    padding="post",
    truncating="post"
)

X_val_integer_300 = pad_sequences(
    X_val_seq300,
    maxlen=MAX_SEQUENCE_LENGTH_300,
    padding="post",
    truncating="post"
)

X_test_integer_300 = pad_sequences(
    X_test_seq300,
    maxlen=MAX_SEQUENCE_LENGTH_300,
    padding="post",
    truncating="post"
)

print("X_train:", X_train_integer_300.shape)
print("X_val  :", X_val_integer_300.shape)
print("X_test :", X_test_integer_300.shape)

X_train: (39665, 300)
X_val  : (4958, 300)
X_test : (4959, 300)


In [71]:
lstm_seq300_model = Sequential([
    Input(
        shape=(300,),
        dtype="int32"
    ),
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    LSTM(128),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])
lstm_seq300_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

lstm_seq300_model.summary()

Model: "sequential_11"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_11 (Embedding)        │ (None, 300, 128)       │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_11 (LSTM)                  │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,979,905 (15.18 MB)

 Trainable params: 3,979,905 (15.18 MB)

 Non-trainable params: 0 (0.00 B)

In [72]:
history_lstm_seq300 = lstm_seq300_model.fit(
    X_train_integer_300,
    y_train,
    validation_data=(X_val_integer_300, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 14s 21ms/step - accuracy: 0.5308 - loss: 0.6911 - val_accuracy: 0.5446 - val_loss: 0.6873
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 13s 21ms/step - accuracy: 0.5504 - loss: 0.6774 - val_accuracy: 0.5498 - val_loss: 0.6684
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 13s 21ms/step - accuracy: 0.6022 - loss: 0.6372 - val_accuracy: 0.6460 - val_loss: 0.6603
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 13s 20ms/step - accuracy: 0.8417 - loss: 0.3665 - val_accuracy: 0.8762 - val_loss: 0.3038
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 12s 20ms/step - accuracy: 0.9266 - loss: 0.1950 - val_accuracy: 0.8814 - val_loss: 0.3102
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 12s 20ms/step - accuracy: 0.9599 - loss: 0.1180 - val_accuracy: 0.8810 - val_loss: 0.3520
Epoch 7/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 13s 20ms/step - accuracy: 0.9804 - loss: 0.0659 - val_accuracy: 0.8788 - val_loss: 0.3948
Epoch 8/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 13s 20ms/step - accuracy: 0.9906 - loss: 0.0376 - 

In [74]:
lstm_seq300_results, lstm_seq300_probabilities, lstm_seq300_predictions, lstm_seq300_cm = evaluate_model(
    lstm_seq300_model,
    X_test_integer_300,
    y_test,
    "LSTM Sequence Length 300",
    batch_size=64
)

LSTM Sequence Length 300 RESULTS
Accuracy : 0.87457
Precision: 0.88622
Recall   : 0.86059
F1 Score : 0.87322
ROC-AUC  : 0.93523

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.8635    0.8887    0.8759      2470
    Positive     0.8862    0.8606    0.8732      2489

    accuracy                         0.8746      4959
   macro avg     0.8749    0.8746    0.8746      4959
weighted avg     0.8749    0.8746    0.8746      4959

Confusion Matrix
[[2195  275]
 [ 347 2142]]


In [75]:
lstm_results.append(lstm_seq300_results)
lstm_df = pd.DataFrame(lstm_results)

display(lstm_df)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,LSTM Baseline,0.882640,0.879130,0.888310,0.883690,0.940160
1,LSTM EarlyStopping,0.880218,0.886893,0.872640,0.879708,0.948134
2,LSTM Dropout,0.751563,0.886294,0.579349,0.700680,0.834259
3,LSTM Batch Normalization,0.817907,0.744150,0.971073,0.842601,0.945947
4,LSTM Learning Rate 0.0005,0.856019,0.865979,0.843712,0.854701,0.916978
5,LSTM ReduceLR,0.884251,0.885312,0.883889,0.884600,0.943946
6,LSTM Batch Size 32,0.874773,0.851392,0.909200,0.879347,0.932626
7,LSTM Batch Size 128,0.881831,0.881363,0.883487,0.882424,0.940744
8,LSTM SGD,0.501916,0.501922,0.996786,0.667653,0.513804
9,LSTM RMSprop,0.517040,0.715596,0.062676,0.115257,0.529041


In [76]:
lstm_df.to_csv("lstm_model_comparison.csv", index=False)
print("saved")

saved


In [77]:
lstm_reducelr_model.save("lstm_reducelr_best.keras")
print("saved")

saved


In [78]:
with open("lstm_reducelr_history.pkl", "wb") as f:
    pickle.dump(history_lstm_reducelr.history, f)

print("history saved")

history saved
